# Dataset AirQuality

In [40]:
import pandas as pd
import openmeteo_requests
import requests_cache
from retry_requests import retry

In [ ]:
gen24 = pd.read_csv("Data acquisition/gen24.csv")
gen25 = pd.read_csv("Data acquisition/gen25.csv")

SO_MI =  [6958, 6925, 6926, 17293, 6908, 10273, 20429, 6956]

gen24_filtered = gen24[gen24['IdSensore'].isin(SO_MI)]
gen25_filtered = gen25[gen25['IdSensore'].isin(SO_MI)]

gen24_filtered = gen24_filtered.reset_index(drop = True)
gen25_filtered = gen25_filtered.reset_index(drop = True)

station_info = pd.read_csv("Data acquisition/station_info.csv")

quality_25 = pd.merge(gen25_filtered, station_info, on='IdSensore', how='left')
quality_24 = pd.merge(gen24_filtered, station_info, on='IdSensore', how='left')


quality_24['Data'] = pd.to_datetime(quality_24['Data'], format='%d/%m/%Y %I:%M:%S %p').dt.strftime('%Y-%m-%d')
quality_25['Data'] = pd.to_datetime(quality_25['Data'], format='%d/%m/%Y %I:%M:%S %p').dt.strftime('%Y-%m-%d')

# Elenco delle colonne da rimuovere
colonne_da_eliminare = ['Stato', 'idOperatore', 'Storico', 'Location', 'Utm_Nord', 'UTM_Est', 'DataStop', 'NomeTipoSensore', 'UnitaMisura', 'Idstazione', 'Comune', 'DataStart']

# Rimuovo le colonne
quality_24 = quality_24.drop(columns=colonne_da_eliminare)
quality_25 = quality_25.drop(columns=colonne_da_eliminare)

# 1. Pulizia
# Trasformiamo 'lat' e 'lng' da stringhe con virgola (es. "45,52") a numeri (es. 45.52)
quality_24['lat'] = quality_24['lat'].astype(str).str.replace(',', '.').astype(float)
quality_24['lng'] = quality_24['lng'].astype(str).str.replace(',', '.').astype(float)

quality_25['lat'] = quality_25['lat'].astype(str).str.replace(',', '.').astype(float)
quality_25['lng'] = quality_25['lng'].astype(str).str.replace(',', '.').astype(float)

# Rinominiamo subito le colonne (Consistency)
quality_24 = quality_24.rename(columns={'lat': 'latitude', 'lng': 'longitude'})
quality_25 = quality_25.rename(columns={'lat': 'latitude', 'lng': 'longitude'})


# 3. Pulizia Valori Numerici (per la colonna 'Valore' con le virgole)
if quality_24['Valore'].dtype == 'object':
    quality_24['Valore'] = quality_24['Valore'].str.replace(',', '.').astype(float)

if quality_25['Valore'].dtype == 'object':
    quality_25['Valore'] = quality_25['Valore'].str.replace(',', '.').astype(float)

# 4. Uniformazione Coordinate (Arrotondamento a 5 decimali)
quality_24['latitude'] = quality_24['latitude'].round(5)
quality_24['longitude'] = quality_24['longitude'].round(5)
quality_25['latitude'] = quality_25['latitude'].round(5)
quality_25['longitude'] = quality_25['longitude'].round(5)

new_names = {
    'IdSensore': 'sensor_id',
    'Data': 'date',
    'Valore': 'value',
    'NomeStazione': 'station_name',
    'Quota': 'altitude',
    'Provincia': 'province'
}
quality_24 = quality_24.rename(columns=new_names)
quality_25 = quality_25.rename(columns=new_names)

# Conta quanti giorni ci sono per ogni coppia di coordinate
# print(quality_24.groupby(['latitude', 'longitude']).size())
# print(quality_25.groupby(['latitude', 'longitude']).size())

quality_24.to_csv("Data acquisition/quality_24.csv", index=False)
quality_25.to_csv("Data acquisition/quality_25.csv", index=False)
quality_25

,sensor_id,date,value,station_name,altitude,province,latitude,longitude
0,6908,2025-01-01,233.0,Pioltello Limito,114,MI,45.48363,9.32736
1,6925,2025-01-01,31.0,Bormio v.Monte Braulio,1225,SO,46.46952,10.37543
2,6926,2025-01-01,51.0,Morbegno v.Cortivacci,262,SO,46.13549,9.56624
3,6958,2025-01-01,51.0,Sondrio v.Mazzini,307,SO,46.16797,9.87015
4,6956,2025-01-01,143.0,Milano Verziere,116,MI,45.46335,9.19533
...,...,...,...,...,...,...,...,...
707,6956,2025-03-31,10.0,Milano Verziere,116,MI,45.46335,9.19533
708,6958,2025-03-31,3.0,Sondrio v.Mazzini,307,SO,46.16797,9.87015
709,6925,2025-03-31,7.0,Bormio v.Monte Braulio,1225,SO,46.46952,10.37543
710,6926,2025-03-31,11.0,Morbegno v.Cortivacci,262,SO,46.13549,9.56624


# API Open-Meteo

# 2024

In [ ]:
# 1. Setup dell'API client con cache e retry
# 1. Setup API
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# 2. Le tue 8 coordinate precise
lats = [46.16797103, 46.46952397, 46.135486, 46.16785664, 
        45.48363242, 45.46334886, 45.49631883, 45.478347]
lngs = [9.87014581, 10.37543275, 9.56624, 9.87921169, 
        9.32736196, 9.19532511, 9.19093444, 9.231667]

url = "https://archive-api.open-meteo.com/v1/archive"
params24 = {
    "latitude": lats,
    "longitude": lngs,
    "start_date": "2024-01-01",
    "end_date": "2024-03-31",
    "daily": ["precipitation_sum", "precipitation_hours", "temperature_2m_min", "wind_speed_10m_max"],
    "timezone": "Europe/Berlin",
}

# 3. Chiamata API
responses24 = openmeteo.weather_api(url, params=params24)

# 4. Elaborazione di TUTTE le risposte
all_stations_list24 = []

for i, response in enumerate(responses24):
    daily24 = response.Daily()
    
    # 1. Decodifica il fuso orario (risolve l'errore 'bytes')
    tz = response.Timezone().decode('utf-8')
    start_utc = pd.to_datetime(daily24.Time(), unit="s", utc=True)
    start_local = start_utc.tz_convert(tz)
    
    # 2. Crea il range di date (senza aggiungere .date alla fine!)
    date_range24 = pd.date_range(
        start=start_local,
        periods=len(daily24.Variables(0).ValuesAsNumpy()),
        freq="D"
    )

    # Estrazione dati della stazione i-esima
    data24 = {
        "date": date_range24.strftime('%Y-%m-%d'),
        "latitude": lats[i],   # <--- Forza la TUA latitudine originale
        "longitude": lngs[i],  # <--- Forza la TUA longitudine originale
        "temperature_min": daily24.Variables(2).ValuesAsNumpy(),
        "wind_speed_max": daily24.Variables(3).ValuesAsNumpy(),
        "precipitation_sum": daily24.Variables(0).ValuesAsNumpy(),
        "precipitation_hours": daily24.Variables(1).ValuesAsNumpy(),
    }
    
    # Crea un dataframe temporaneo per questa stazione e aggiungilo alla lista
    df_stazione24 = pd.DataFrame(data24)
    all_stations_list24.append(df_stazione24)

# 5. UNIONE FINALE (fondamentale: deve essere fuori dal ciclo for)
daily_weather24 = pd.concat(all_stations_list24, ignore_index=True)

# 6. PREPARAZIONE METEO PER IL MERGE

# Arrotondiamo le coordinate del meteo a 5 cifre (Accuracy)
daily_weather24['latitude'] = daily_weather24['latitude'].round(5)
daily_weather24['longitude'] = daily_weather24['longitude'].round(5)

# Rimuoviamo il fuso orario (timezone) dalla data del meteo per renderla compatibile (Consistency)
# Spesso l'API restituisce UTC, noi vogliamo confrontarla con la data locale dell'aria
daily_weather24['date'] = pd.to_datetime(daily_weather24['date']).dt.tz_localize(None)

daily_weather24.to_csv("Data acquisition/weather_24_pm10.csv", index=False)
daily_weather24

,date,latitude,longitude,temperature_min,wind_speed_max,precipitation_sum,precipitation_hours
0,2024-01-01,46.16797,9.87015,-2.351,5.154416,0.100000,1.0
1,2024-01-02,46.16797,9.87015,0.849,5.495161,0.100000,1.0
2,2024-01-03,46.16797,9.87015,3.599,5.191994,0.000000,0.0
3,2024-01-04,46.16797,9.87015,1.799,5.483356,0.000000,0.0
4,2024-01-05,46.16797,9.87015,2.199,5.506941,26.000002,13.0
...,...,...,...,...,...,...,...
723,2024-03-27,45.47835,9.23167,6.500,21.370783,26.000002,16.0
724,2024-03-28,45.47835,9.23167,4.650,10.594036,4.900000,8.0
725,2024-03-29,45.47835,9.23167,7.950,18.584511,4.200000,14.0
726,2024-03-30,45.47835,9.23167,8.900,33.129780,8.299999,7.0


# 2025

In [ ]:
# 1. Setup dell'API client con cache e retry
# 1. Setup API
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# 2. Le tue 8 coordinate precise
lats = [46.16797103, 46.46952397, 46.135486, 46.16785664, 
        45.48363242, 45.46334886, 45.49631883, 45.478347]
lngs = [9.87014581, 10.37543275, 9.56624, 9.87921169, 
        9.32736196, 9.19532511, 9.19093444, 9.231667]

url = "https://archive-api.open-meteo.com/v1/archive"
params25 = {
    "latitude": lats,
    "longitude": lngs,
    "start_date": "2025-01-01",
    "end_date": "2025-03-31",
    "daily": ["precipitation_sum", "precipitation_hours", "temperature_2m_min", "wind_speed_10m_max"],
    "timezone": "Europe/Berlin",
}

# 3. Chiamata API
responses25 = openmeteo.weather_api(url, params=params25)

# 4. Elaborazione di TUTTE le risposte
all_stations_list25 = []

for i, response in enumerate(responses25):
    daily25 = response.Daily()
    
    # 1. Decodifica il fuso orario (risolve l'errore 'bytes')
    tz = response.Timezone().decode('utf-8')
    start_utc = pd.to_datetime(daily25.Time(), unit="s", utc=True)
    start_local = start_utc.tz_convert(tz)
    
    # 2. Crea il range di date (senza aggiungere .date alla fine!)
    date_range25 = pd.date_range(
        start=start_local,
        periods=len(daily25.Variables(0).ValuesAsNumpy()),
        freq="D"
    )

    # Estrazione dati della stazione i-esima
    data25 = {
        "date": date_range25.strftime('%Y-%m-%d'),
        "latitude": lats[i],   # <--- Forza la TUA latitudine originale
        "longitude": lngs[i],  # <--- Forza la TUA longitudine originale
        "temperature_min": daily25.Variables(2).ValuesAsNumpy(),
        "wind_speed_max": daily25.Variables(3).ValuesAsNumpy(),
        "precipitation_sum": daily25.Variables(0).ValuesAsNumpy(),
        "precipitation_hours": daily25.Variables(1).ValuesAsNumpy(),
    }
    
    # Crea un dataframe temporaneo per questa stazione e aggiungilo alla lista
    df_stazione25 = pd.DataFrame(data25)
    all_stations_list25.append(df_stazione25)

# 5. UNIONE FINALE (fondamentale: deve essere fuori dal ciclo for)
daily_weather25 = pd.concat(all_stations_list25, ignore_index=True)

# 6. PREPARAZIONE METEO PER IL MERGE

# Arrotondiamo le coordinate del meteo a 5 cifre (Accuracy)
daily_weather25['latitude'] = daily_weather25['latitude'].round(5)
daily_weather25['longitude'] = daily_weather25['longitude'].round(5)

# Rimuoviamo il fuso orario (timezone) dalla data del meteo per renderla compatibile (Consistency)
# Spesso l'API restituisce UTC, noi vogliamo confrontarla con la data locale dell'aria
daily_weather25['date'] = pd.to_datetime(daily_weather25['date']).dt.tz_localize(None)

daily_weather25.to_csv("Data acquisition/weather_25_pm10.csv", index=False)
daily_weather25

,date,latitude,longitude,temperature_min,wind_speed_max,precipitation_sum,precipitation_hours
0,2025-01-01,46.16797,9.87015,-0.551000,1.310420,0.0,0.0
1,2025-01-02,46.16797,9.87015,-1.250999,3.494968,0.2,2.0
2,2025-01-03,46.16797,9.87015,-4.601000,4.669604,0.7,5.0
3,2025-01-04,46.16797,9.87015,-5.151000,4.532284,0.0,0.0
4,2025-01-05,46.16797,9.87015,-2.500999,3.396233,0.1,1.0
...,...,...,...,...,...,...,...
715,2025-03-27,45.47835,9.23167,8.150000,8.536252,0.0,0.0
716,2025-03-28,45.47835,9.23167,8.850000,7.380000,0.0,0.0
717,2025-03-29,45.47835,9.23167,10.300000,8.209263,2.1,7.0
718,2025-03-30,45.47835,9.23167,8.350000,16.663374,0.0,0.0


# Data Integration Mongo DB (join dati)

In [44]:
# lookup
{
  from: "Weather_24", // Assicurati che il nome sia esatto
  let: { q_date: "$Data", q_lat: "$latitude", q_lon: "$longitude" },
  pipeline: [
    { $match: {
        $expr: {
          $and: [
            { $eq: ["$Data", "$$q_date"] },
            { $eq: ["$latitude", "$$q_lat"] },
            { $eq: ["$longitude", "$$q_lon"] }
          ]
        }
      }
    }
  ],
  as: "meteo_join"
}

SyntaxError: invalid syntax (3986443739.py, line 3)

In [ ]:
# unwind
{
  path: "$meteo_join",
  preserveNullAndEmptyArrays: false
}

# Pulizia Dataset Completo (dopo il join)

In [ ]:
dataset_completo = pd.read_csv("Data Integration/pm10_24_25.csv")

# 1. Rinominiamo le colonne del meteo (Questo va bene)
nuovi_nomi = {
    'weather.temperature_min': 'temperature_min',
    'weather.wind_speed_max': 'wind_speed_max',
    'weather.precipitation_sum': 'precipitation_sum',
    'weather.precipitation_hours': 'precipitation_hours'
}
dataset_completo = dataset_completo.rename(columns=nuovi_nomi)

# 2. Unifichiamo Latitude e Longitude e togliamo gli ID corretti
# NOTA: Ho tolto .$oid e .$date perché nel CSV non ci sono!
colonne_da_eliminare = [
    'weather.latitude', 
    'weather.longitude', 
    'weather._id',       # Nome corretto nel CSV
    'weather.date',      # Nome corretto nel CSV
    '_id'                   # Nome corretto nel CSV
]

dataset_completo = dataset_completo.drop(columns=[c for c in colonne_da_eliminare if c in dataset_completo.columns])


# 4. Gestione Data e Anno
# La colonna si chiama già 'Data', non serve rinominarla. La convertiamo solo.
dataset_completo['date'] = pd.to_datetime(dataset_completo['date']).dt.tz_localize(None)
dataset_completo['year'] = dataset_completo['date'].dt.year

dataset_completo.to_csv("pm10.csv")
dataset_completo

,sensor_id,date,value,station_name,altitude,province,latitude,longitude,temperature_min,wind_speed_max,precipitation_sum,precipitation_hours,year
0,6958,2024-01-01,15.0,Sondrio v.Mazzini,307,SO,46.16797,9.87015,-2.351000,5.154416,0.1,1,2024
1,6925,2024-01-01,15.0,Bormio v.Monte Braulio,1225,SO,46.46952,10.37543,-13.617001,9.957108,2.8,2,2024
2,6926,2024-01-01,8.0,Morbegno v.Cortivacci,262,SO,46.13549,9.56624,0.260500,7.235910,0.1,1,2024
3,17293,2024-01-01,16.0,Sondrio v.Paribelli,290,SO,46.16786,9.87921,-2.279500,5.154416,0.1,1,2024
4,6908,2024-01-01,33.0,Pioltello Limito,114,MI,45.48363,9.32736,1.250000,12.429127,0.0,0,2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1427,6956,2025-03-31,10.0,Milano Verziere,116,MI,45.46335,9.19533,10.450000,24.696104,0.0,0,2025
1428,6958,2025-03-31,3.0,Sondrio v.Mazzini,307,SO,46.16797,9.87015,6.199000,6.037383,0.0,0,2025
1429,6925,2025-03-31,7.0,Bormio v.Monte Braulio,1225,SO,46.46952,10.37543,-0.767000,14.058450,0.9,7,2025
1430,6926,2025-03-31,11.0,Morbegno v.Cortivacci,262,SO,46.13549,9.56624,7.060500,9.541697,0.0,0,2025


# Codice Per Grafico 1 e 2 (boxplot)

In [46]:
# 2. Filtriamo solo i dati dell'anno 2024
df_2024 = dataset_completo[dataset_completo['year'] == 2024].copy()

# 3. Selezioniamo solo le colonne necessarie
# 'province' per l'asse X, 'value' per l'asse Y
boxplot_data24 = df_2024[['province', 'value']]

# 4. Rinominiamo le colonne per chiarezza in Flourish
boxplot_data24.columns = ['Province', 'PM10']

# 5. Salvataggio del file CSV
boxplot_data24.to_csv('boxplot_pm10_2024.csv', index=False)

In [ ]:
# 2. Filtriamo solo i dati dell'anno 2024
df_2025 = dataset_completo[dataset_completo['year'] == 2025].copy()

# 3. Selezioniamo solo le colonne necessarie
# 'province' per l'asse X, 'value' per l'asse Y
boxplot_data25 = df_2025[['province', 'value']]

# 4. Rinominiamo le colonne per chiarezza in Flourish
boxplot_data25.columns = ['Province', 'PM10']

# 5. Salvataggio del file CSV
boxplot_data24.to_csv("graphs/boxplot_pm10_2024.csv", index=False)

# Codice Per Grafico 5 e 6 (line&bar chart)

## Andamento Precipitazioni 24/25 & Andamento PM10 24/25 - MILANO

In [ ]:
dataset_completo['date'] = pd.to_datetime(dataset_completo['date'])

# --- FILTRO MILANO ---
# Selezioniamo solo i dati relativi alla provincia di Milano
milano_data = dataset_completo[dataset_completo['province'] == 'MI'].copy()

# 2. Calcolo medie giornaliere per Milano
daily_stats = milano_data.groupby(['date', 'year']).agg({
    'value': 'mean',
    'precipitation_sum': 'mean'
}).reset_index()

# 3. Creiamo la chiave di allineamento (SOLO Giorno-Mese)
daily_stats['Giorno_Mese'] = daily_stats['date'].dt.strftime('%d-%m')

# 4. Pivot corretta
# Usiamo solo 'Giorno_Mese' come indice per allineare 2024 e 2025
pivot_df = daily_stats.pivot(
    index='Giorno_Mese', 
    columns='year', 
    values=['value', 'precipitation_sum']
).reset_index()

# 5. Pulizia nomi colonne per Flourish
pivot_df.columns = ['Giorno', 'PM10_2024', 'PM10_2025', 'Precipitations_2024', 'Precipitations_2025']

# 6. Ordinamento Cronologico Corretto
pivot_df['temp_date'] = pd.to_datetime(pivot_df['Giorno'] + '-2024', format='%d-%m-%Y')
pivot_df = pivot_df.sort_values('temp_date').drop(columns=['temp_date'])

# 7. Salvataggio del file specifico per Milano
pivot_df.to_csv("graphs/smog_killer_sondrio_comparison.csv", index=False)


## Andamento Precipitazioni 24/25 & Andamento PM10 24/25 - SONDRIO

In [49]:
dataset_completo['date'] = pd.to_datetime(dataset_completo['date'])

# --- FILTRO MILANO ---
# Selezioniamo solo i dati relativi alla provincia di Milano
sondrio_data = dataset_completo[dataset_completo['province'] == 'SO'].copy()

# 2. Calcolo medie giornaliere per Milano
daily_stats = sondrio_data.groupby(['date', 'year']).agg({
    'value': 'mean',
    'precipitation_sum': 'mean'
}).reset_index()

# 3. Creiamo la chiave di allineamento (SOLO Giorno-Mese)
daily_stats['Giorno_Mese'] = daily_stats['date'].dt.strftime('%d-%m')

# 4. Pivot corretta
# Usiamo solo 'Giorno_Mese' come indice per allineare 2024 e 2025
pivot_df = daily_stats.pivot(
    index='Giorno_Mese', 
    columns='year', 
    values=['value', 'precipitation_sum']
).reset_index()

# 5. Pulizia nomi colonne per Flourish
pivot_df.columns = ['Giorno', 'PM10_2024', 'PM10_2025', 'Precipitations_2024', 'Precipitations_2025']

# 6. Ordinamento Cronologico Corretto
pivot_df['temp_date'] = pd.to_datetime(pivot_df['Giorno'] + '-2024', format='%d-%m-%Y')
pivot_df = pivot_df.sort_values('temp_date').drop(columns=['temp_date'])

# 7. Salvataggio del file specifico per Milano
pivot_df.to_csv('smog_killer_sondrio_comparison.csv', index=False)